# tabular_agent — Phase 1 Core Library Walkthrough

This notebook explains every module of the Phase-1 core library: what it does, why it is
designed the way it is, and how the pieces fit together. Each section shows the relevant
source code with commentary, followed by a small runnable example.

The library is a **leakage-safe tabular-ML core**: a minimal set of composable functions for
training gradient-boosted models on tabular data with honest, cross-validated evaluation.
In Phase 2 it will be wrapped by an MCP server so an agent can drive the workflow tool-by-tool.

Module map, in the order data flows through them:

| Module | Responsibility |
|---|---|
| `io` | Load/save tables (CSV, Parquet) |
| `taskspec` | Declare the task; auto-detect whatever the user left out |
| `inspect` | Profile a DataFrame so an agent can reason about it |
| `runctx` | The run directory *is* the state |
| `cv` | Deterministic fold generation, shared by encoding and training |
| `encoders` | Leakage-safe target encoding |
| `metrics` | Metric registry with a single prediction-format contract |
| `models` | Model wrappers + cross-validated training with OOF predictions |
| `blend` | Non-negative ensemble weights optimized on OOF predictions |
| `report` | Human-readable run report with a baseline-honesty warning |
| `pipeline` | Serializable feature pipeline + predictor bundle |

> Code snippets in markdown cells are copied from the source for readability; the runnable
> cells import the real library.

## 0. Setup: a small synthetic dataset

A churn-style binary-classification dataset used throughout: two numeric features
(`age`, `income`), one categorical feature (`city`) that genuinely carries signal, and a few
missing values in `income`. LightGBM/XGBoost handle NaNs natively, so we leave them in.

In [1]:
import json
import os
import tempfile

import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 3000

city = rng.choice(["berlin", "hamburg", "munich", "cologne", "leipzig"],
                  size=n, p=[0.30, 0.25, 0.20, 0.15, 0.10])
city_effect = {"berlin": 0.8, "hamburg": -0.5, "munich": 1.2, "cologne": -1.0, "leipzig": 0.2}
age = rng.normal(40, 12, n).round(1)
income = rng.lognormal(10, 0.4, n).round(0)

logit = 0.04 * (age - 40) + np.array([city_effect[c] for c in city]) + rng.normal(0, 1.0, n)
churn = (logit > 0.3).astype(int)

df = pd.DataFrame({"age": age, "income": income, "city": city, "churn": churn})
df.loc[rng.choice(n, 15, replace=False), "income"] = np.nan

print(f"{len(df)} rows, churn rate {df['churn'].mean():.2f}")
df.head()

3000 rows, churn rate 0.49


,age,income,city,churn
0,37.0,15591.0,munich,1
1,35.4,22192.0,berlin,0
2,29.5,33603.0,berlin,1
3,25.7,34258.0,berlin,0
4,40.7,28594.0,cologne,0


## 1. `io` — table loading by file extension

```python
def load_table(path: str) -> pd.DataFrame:
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"unsupported extension: {path}")

def save_parquet(df: pd.DataFrame, path: str) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    df.to_parquet(path, index=False)
    return path
```

Three deliberate choices:

- **Dispatch on the extension, nothing clever.** The caller (later: the agent) says exactly
  what it wants; anything unknown raises immediately instead of guessing.
- **`save_parquet` creates parent directories** so callers never have to `mkdir` — the same
  convention `RunContext.path` follows (section 4).
- **`index=False`**: the row index is not data. Dropping it makes save→load round-trips
  stable instead of accumulating `Unnamed: 0`-style artifacts.

In [2]:
from tabular_agent.io import load_table, save_parquet

tmp = tempfile.mkdtemp()
path = save_parquet(df, os.path.join(tmp, "data", "churn.parquet"))
df_back = load_table(path)
print("round-trip OK:", df_back.shape == df.shape)

try:
    load_table("data.xlsx")
except ValueError as e:
    print("as expected:", e)

round-trip OK: True
as expected: unsupported extension: data.xlsx


## 2. `taskspec` — the single source of truth

Two dataclasses with a clear division of labor:

- **`TaskSpec`** is what the user *declares*. Only `target` is required; `task_type`,
  `metric` etc. may be `None`.
- **`ResolvedSpec`** is what the rest of the library *consumes*. No `Optional` fields left —
  downstream code never guesses.

`resolve_taskspec` bridges the two: it fills every gap with auto-detection, once, in one
place.

```python
def detect_task_type(y: pd.Series) -> str:
    n_unique = int(y.nunique(dropna=True))
    if n_unique <= 2:
        return "binary"
    if pd.api.types.is_float_dtype(y):
        return "regression"
    if pd.api.types.is_integer_dtype(y):
        return "multiclass" if n_unique <= 20 else "regression"
    return "multiclass"
```

The heuristic, top to bottom:

1. **≤ 2 unique values → binary**, regardless of dtype.
2. **Floats → regression** (float targets with few unique values are rare in practice).
3. **Integers are ambiguous**: few unique values look like class labels (→ multiclass),
   many look like counts or amounts (→ regression). The cut-off is 20.
4. **Everything else** (strings, categoricals) → multiclass.

```python
def default_metric(task_type: str, y: pd.Series) -> str:
    if task_type == "regression":
        if pd.api.types.is_numeric_dtype(y) and bool((y.dropna() >= 0).all()):
            return "rmsle"
        return "rmse"
    return "logloss"
```

- Non-negative regression targets default to **RMSLE**: it measures *relative* error, which
  suits skewed positive quantities (prices, counts) far better than RMSE.
- Classification defaults to **logloss**, a proper scoring rule: it rewards calibrated
  probabilities, whereas accuracy throws probability information away.

In [3]:
from tabular_agent.taskspec import TaskSpec, detect_task_type, default_metric, resolve_taskspec

examples = {
    "0/1 ints": pd.Series([0, 1, 0, 1]),
    "3 string labels": pd.Series(["a", "b", "c", "a"]),
    "ints, 5 unique": pd.Series([1, 2, 3, 4, 5] * 4),
    "ints, 100 unique": pd.Series(range(100)),
    "floats": pd.Series([1.5, 2.7, 3.1, 4.9]),
    "floats w/ negatives": pd.Series([-1.5, 2.7, 3.1, 4.9]),
}
for name, s in examples.items():
    t = detect_task_type(s)
    print(f"{name:22s} -> {t:11s} (default metric: {default_metric(t, s)})")

0/1 ints               -> binary      (default metric: logloss)
3 string labels        -> multiclass  (default metric: logloss)
ints, 5 unique         -> multiclass  (default metric: logloss)
ints, 100 unique       -> regression  (default metric: rmsle)
floats                 -> regression  (default metric: rmsle)
floats w/ negatives    -> regression  (default metric: rmse)


In [4]:
spec = TaskSpec(target="churn")          # user only declares the target
resolved = resolve_taskspec(spec, df["churn"])
resolved

ResolvedSpec(target='churn', task_type='binary', metric='logloss', positive_class=None, notes=None, n_splits=5, seed=42)

## 3. `inspect` — dataset profiling

`profile_dataframe` produces a small, JSON-serializable summary: per-column dtype, missing
count and unique count, plus a target block. It is deliberately cheap and structured — this
is the payload the Phase-2 `inspect_data` tool returns so the *agent* can decide things like
"`city` has 5 unique values → target-encode it" or "`income` has missing values → note it".

In [5]:
from tabular_agent.inspect import profile_dataframe

print(json.dumps(profile_dataframe(df, target="churn"), indent=2))

{
  "n_rows": 3000,
  "n_cols": 4,
  "columns": {
    "age": {
      "dtype": "float64",
      "n_missing": 0,
      "n_unique": 563
    },
    "income": {
      "dtype": "float64",
      "n_missing": 15,
      "n_unique": 2852
    },
    "city": {
      "dtype": "str",
      "n_missing": 0,
      "n_unique": 5
    },
    "churn": {
      "dtype": "int64",
      "n_missing": 0,
      "n_unique": 2
    }
  },
  "target": {
    "name": "churn",
    "dtype": "int64",
    "n_unique": 2,
    "n_missing": 0
  }
}


## 4. `runctx` — the run directory is the state

```python
class RunContext:
    def __init__(self, run_dir: str):
        self.run_dir = run_dir
        os.makedirs(run_dir, exist_ok=True)

    def path(self, *parts: str) -> str:
        full = os.path.join(self.run_dir, *parts)
        os.makedirs(os.path.dirname(os.path.abspath(full)), exist_ok=True)
        return full
```

Tiny on purpose. The design principle: **there is no in-memory session object** — every
artifact of a run (data snapshots, models, the report, the predictor bundle) is a file
inside one run directory. That makes a run:

- **restartable** — nothing is lost when the process dies,
- **inspectable** — `ls run_dir` tells you what happened,
- **a natural fit for an MCP server**, where every tool call is stateless and the run
  directory is the only thing passed between calls.

`path()` creates parent directories as a side effect, so callers just ask for a path and
write to it.

In [7]:
from tabular_agent.runctx import RunContext

run_ctx = RunContext(os.path.join(tmp, "run-001"))
p = run_ctx.path("folds", "fold_0.json")     # nested dirs created automatically
print("ready to write:", p)
print("exists:", os.path.isdir(os.path.dirname(p)))

/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/tmp8x8yefdp
ready to write: /var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/tmp8x8yefdp/run-001/folds/fold_0.json
exists: True


## 5. `cv` — one fold split to rule them all

```python
def make_folds(y, task_type: str, n_splits: int = 5, seed: int = 42):
    y = np.asarray(y)
    if task_type in ("binary", "multiclass"):
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return [(tr, va) for tr, va in splitter.split(np.zeros(len(y)), y)]
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in splitter.split(np.zeros(len(y)))]
```

- **Stratified for classification**: each fold keeps the class proportions, which stabilizes
  fold metrics, especially with imbalanced targets. Plain `KFold` for regression.
- **`np.zeros(len(y))` as X**: the splitters only need the number of rows (and `y` for
  stratification) to generate *index* arrays — features are irrelevant here.
- **`shuffle=True` + fixed seed** → the split is deterministic and reproducible.

The important design decision is that this lives in *its own* module and folds are passed
around **explicitly**: target encoding (section 6) and model training (section 8) must use
the **same partition**. If the encoder used different folds than the trainer, a row that is
"validation" for the model could have contributed its target to the encoding statistics —
silent leakage. One shared fold artifact makes that impossible.

In [7]:
from tabular_agent.cv import make_folds

folds = make_folds(df["churn"], "binary", n_splits=5, seed=42)

for i, (tr, va) in enumerate(folds):
    print(f"fold {i}: train={len(tr)}, val={len(va)}, "
          f"churn rate in val={df['churn'].to_numpy()[va].mean():.3f}")

folds_again = make_folds(df["churn"], "binary", n_splits=5, seed=42)
print("deterministic:", all(np.array_equal(a[1], b[1]) for a, b in zip(folds, folds_again)))

fold 0: train=2400, val=600, churn rate in val=0.490
fold 1: train=2400, val=600, churn rate in val=0.490
fold 2: train=2400, val=600, churn rate in val=0.488
fold 3: train=2400, val=600, churn rate in val=0.488
fold 4: train=2400, val=600, churn rate in val=0.488
deterministic: True


## 6. `encoders` — leakage-safe target encoding

### 6.1 The problem

High-cardinality categoricals (cities, product IDs, ...) don't fit one-hot encoding (column
explosion) and label encoding imposes a fake ordering. **Target encoding** replaces each
category with a statistic of the target within that category — one informative numeric
column. But it is the most leakage-prone transform in tabular ML, which is why this module
is built the way it is.

### 6.2 Smoothing — don't trust rare categories

The core of `TargetEncoder.fit`:

```python
smooth = (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing) / (
    stats["count"] + self.smoothing
)
```

This is a **weighted average between the category mean and the global mean**. With
`smoothing=10` (the default), a category with 10 observations is mixed 50/50; a category
with 2 observations is pulled almost entirely to the global mean. Rare categories have
noisy means — shrinking them toward the global mean protects against overfitting.

Other implementation details worth noting:

- `fit` stores only **plain parameters** (`mapping_`: category → value, `global_mean_`).
- `transform` maps **unseen categories to the global mean** — the only honest guess for a
  category never observed during fit.
- `to_dict` / `from_dict` make the encoder a pure-parameter object, JSON-serializable for
  the predictor bundle (section 11). `ENCODER_CLASSES` is the registry `Pipeline` uses to
  rehydrate the right class from `"kind"`.

In [8]:
from tabular_agent.encoders import TargetEncoder

demo = pd.DataFrame({"cat": ["common"] * 60 + ["mid"] * 10 + ["rare"] * 2})
y_demo = np.r_[rng.normal(1.0, 0.2, 60), rng.normal(2.0, 0.2, 10), [5.0, 5.2]]

enc = TargetEncoder(["cat"], smoothing=10.0).fit(demo, y_demo)

pd.DataFrame({
    "count": demo["cat"].value_counts(),
    "raw mean": pd.Series(y_demo).groupby(demo["cat"]).mean(),
    "encoded (smoothed)": pd.Series(enc.mapping_["cat"]),
}).round(3)

,count,raw mean,encoded (smoothed)
common,60,0.983,1.019
mid,10,1.975,1.605
rare,2,5.100,1.879


The `rare` category has a raw mean of ~5.1, but only 2 observations — its encoded value is
shrunk most of the way back toward the global mean (~1.25). `common`, with 60 observations,
keeps almost its raw mean.

### 6.3 The leakage trap, and out-of-fold encoding

Naive recipe: fit the encoder on **all** rows, then cross-validate a model on the encoded
feature. This is wrong: each row's own target is inside the mapping used to encode it, so
the feature *secretly contains the label* and the CV score is inflated.

`oof_target_encode` fixes this with the same fold logic used for model training:

```python
def oof_target_encode(df, y, columns, folds, smoothing: float = 10.0):
    ...
    for tr, va in folds:
        enc = TargetEncoder(columns, smoothing).fit(df.iloc[tr], y[tr])   # train rows only
        transformed = enc.transform(df.iloc[va])                          # fill val rows
        ...
    final = TargetEncoder(columns, smoothing).fit(df, y)                  # for inference
    return oof, final
```

- Every row's encoded value is computed **without that row's target** (it was in the
  validation split when its value was filled in).
- The **`final` encoder**, fit on all rows, is returned separately: at inference time on
  truly new data there is no leakage concern, so we use all available information. This
  OOF-features-for-training / full-encoder-for-inference asymmetry reappears in the bundle
  (section 11).

### 6.4 Watching leakage happen

The cleanest demonstration: a **pure-noise target**. Nothing can genuinely predict it —
anything above AUC 0.5 is leakage. We keep the demo model-free and simply measure the AUC
of the encoded feature *itself* against the target: does the feature already contain the
label? (`smoothing=1.0` makes the effect vivid — less shrinkage means more of each row's
own target survives in its encoding.)

In [9]:
from tabular_agent.encoders import oof_target_encode
from tabular_agent.metrics import compute_metric

rng2 = np.random.default_rng(1)
n_cats = 60
noise_df = pd.DataFrame({"cat": np.repeat([f"c{i}" for i in range(n_cats)], 600 // n_cats)})
y_noise = rng2.integers(0, 2, 600)          # pure noise: no real signal
folds_noise = make_folds(y_noise, "binary")

# naive: encoder fit on ALL rows -> each row's own target is inside its feature
naive_enc = TargetEncoder(["cat"], smoothing=1.0).fit(noise_df, y_noise)
f_naive = naive_enc.transform(noise_df)["cat_te"].to_numpy()

# out-of-fold: each row encoded without its own target
X_oof, _ = oof_target_encode(noise_df, y_noise, ["cat"], folds_noise, smoothing=1.0)
f_oof = X_oof["cat_te"].to_numpy()

print(f"AUC(feature, target), naive encoding: {compute_metric('auc', y_noise, f_naive):.3f}  <- leakage!")
print(f"AUC(feature, target), OOF encoding  : {compute_metric('auc', y_noise, f_oof):.3f}  <- honest (~0.5)")

AUC(feature, target), naive encoding: 0.679  <- leakage!
AUC(feature, target), OOF encoding  : 0.492  <- honest (~0.5)


The naively encoded feature "predicts" a target that is *pure noise* with AUC ≈ 0.68 —
that skill is fabricated: the feature partially **is** the label. The OOF-encoded feature
sits at chance level, as it must. This single comparison is the reason the library forces
folds to be created first and passed everywhere.

### 6.5 An honest footnote: first-order vs. second-order leakage

Why does this demo compare the feature *directly* against the target instead of
cross-validating a model on it? Because a model-based version has a second, much subtler
channel that would muddy the picture.

OOF encoding guarantees that a row's **own** target is never inside its **own** feature
value. It does *not* guarantee that the **training rows'** features are free of information
about the **validation rows'** targets. Concretely, with 5 shared folds: a training row
from fold 2 got its encoding from "all folds except fold 2" — which *includes* fold 1. So
when `train_cv` evaluates fold 1, the model was trained on features that already have
fold 1's targets mixed in (diluted, via category means). The chain
*val targets → training features → fitted model → val predictions* breaks the independence
that makes AUC = 0.5 on noise.

Empirically (this dataset, 15 seeds):

| Setup | mean AUC on pure noise |
|---|---|
| naive encoding, feature vs. target | ≈ 0.67 — **first-order leakage** (own target in own feature) |
| OOF encoding + `train_cv` on the *same* folds | ≈ 0.52 — **second-order optimism** (val targets in training features) |
| OOF encoding, feature vs. target | ≈ 0.49 — clean |

The library's design choices reflect this hierarchy: first-order leakage (the big one) is
**eliminated** by OOF encoding with shared folds; the small second-order effect is
**accepted** — removing it would require nested cross-validation (folds within folds) at a
multiple of the compute cost, and on real data with real signal it is negligible. It only
becomes visible in this deliberately pathological setup: a pure-noise target and a single
high-cardinality feature.

## 7. `metrics` — one registry, one prediction-format contract

Six metrics behind a single entry point:

```python
SUPPORTED_METRICS = {"rmse", "rmsle", "mae", "logloss", "auc", "accuracy"}

def compute_metric(name, y_true, y_pred, labels=None) -> float: ...
def greater_is_better(name: str) -> bool: ...
```

The important part is the **prediction-format contract** — every caller passes the same
object regardless of which metric is computed:

| Task | `y_pred` format |
|---|---|
| regression | 1-D array of point predictions |
| binary | 1-D array of P(class 1) |
| multiclass | 2-D `(n, k)` probability matrix, columns = sorted classes |

Metrics adapt themselves to the contract instead of the caller adapting to the metric:
`accuracy` derives hard labels internally (argmax for 2-D, 0.5-threshold for 1-D), and
`logloss` accepts a `labels` argument for the fold-missing-a-class case (section 8).

`greater_is_better` encodes each metric's orientation; it drives both the sign flip in the
blend optimizer (section 9) and the baseline comparison in the report (section 10).
`rmsle` refuses negative values because `log1p` would be meaningless there.

In [10]:
from tabular_agent.metrics import compute_metric, greater_is_better

y_true_bin = np.array([0, 1, 1, 0])
p_bin = np.array([0.1, 0.8, 0.6, 0.4])
print("auc     :", compute_metric("auc", y_true_bin, p_bin))
print("accuracy:", compute_metric("accuracy", y_true_bin, p_bin), "(thresholded at 0.5)")

proba_3c = np.array([[0.8, 0.1, 0.1], [0.2, 0.5, 0.3], [0.1, 0.2, 0.7]])
print("accuracy (multiclass):", compute_metric("accuracy", [0, 1, 2], proba_3c), "(argmax)")

print("rmse    :", round(compute_metric("rmse", [1.0, 2.0], [1.5, 2.5]), 4))
print("greater_is_better('auc') =", greater_is_better("auc"),
      "| greater_is_better('rmse') =", greater_is_better("rmse"))

try:
    compute_metric("rmsle", [-1.0, 2.0], [1.0, 2.0])
except ValueError as e:
    print("as expected:", e)

auc     : 1.0
accuracy: 1.0 (thresholded at 0.5)
accuracy (multiclass): 1.0 (argmax)
rmse    : 0.5
greater_is_better('auc') = True | greater_is_better('rmse') = False
as expected: rmsle requires non-negative targets and predictions


## 8. `models` — wrappers, baseline, and cross-validated training

### 8.1 `_Baseline` — the honesty anchor

```python
class _Baseline:
    """Predicts the training mean (regression) or class priors (classification)."""
```

It predicts the train mean / class priors for every row — the best you can do while
ignoring the features entirely. Its job is to be the **floor**: any real model must beat
it, and the report (section 10) warns loudly when one doesn't, because that almost always
means a bug, leakage, or an unlearnable target.

### 8.2 `_make_model` — string → estimator

`"baseline" | "lgbm" | "xgb" | "catboost"` → configured estimator. Libraries are imported
**lazily** inside each branch, so you only pay the import cost for the library you use.
All models get a unified setup: fixed seed, `n_estimators=200`, silenced logging, and a
`params` dict passed through for overrides — we'll use that below to tame XGBoost, whose
default depth-6 trees produce overconfident probabilities (and therefore poor logloss) on a
dataset this size.

### 8.3 `predict_model` — normalizing every model into the contract

```python
def predict_model(model, X, task_type, classes=None) -> np.ndarray:
    if task_type == "regression":
        return np.asarray(model.predict(X), dtype=float)
    proba = np.asarray(model.predict_proba(X), dtype=float)
    ...
    classes = np.asarray(classes)
    model_classes = np.asarray(model.classes_)
    col_idx = np.searchsorted(classes, model_classes)
    full = np.zeros((len(X), len(classes)))
    full[:, col_idx] = proba
```

The subtle part is the **zero-padding**. During CV, a fold's *train* split can miss a rare
class entirely; that fold's model then emits a probability matrix with fewer columns than
the global class set. `searchsorted` maps the model's own `classes_` into the global sorted
`classes` index space, and missing classes get probability 0 — so every fold contributes a
**consistently shaped** row block to the OOF matrix. (This is the fix from commit
`8c88a13`.)

In [11]:
from tabular_agent.models import _make_model, predict_model   # private, but this is a walkthrough

X3 = pd.DataFrame({"x": rng.normal(size=80)})
y3 = pd.Series([0, 2] * 40)                       # classes 1 is missing entirely

model = _make_model("lgbm", "multiclass", None, seed=42)
model.fit(X3, y3)
print("model.classes_ =", model.classes_)

padded = predict_model(model, X3.head(4), "multiclass", classes=np.array([0, 1, 2]))
print("padded shape   =", padded.shape, " (middle column = missing class 1)")
print(np.round(padded, 3))

model.classes_ = [0 2]
padded shape   = (4, 3)  (middle column = missing class 1)
[[0.667 0.    0.333]
 [0.514 0.    0.486]
 [0.514 0.    0.486]
 [0.425 0.    0.575]]


### 8.4 `train_cv` — the training loop

```python
def train_cv(X, y, folds, task_type, metric, model_type="lgbm", params=None, seed=42):
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y)).reset_index(drop=True)
    ...
    for tr, va in folds:
        model = _make_model(model_type, task_type, params, seed)
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = predict_model(model, X.iloc[va], task_type, classes=classes)
        oof[va] = preds
        fold_metrics.append(compute_metric(metric, y.iloc[va].to_numpy(), preds, **metric_kwargs))
        models.append(model)

    final = _make_model(model_type, task_type, params, seed)
    final.fit(X, y)
    return TrainResult(oof, fold_metrics, models, final, list(X.columns), model_type)
```

Line by line:

- **`reset_index(drop=True)`**: folds are *positional* index arrays. Resetting guarantees
  `X.iloc[tr]` and `oof[va]` refer to the same rows even if the caller passes a filtered
  DataFrame with a gappy index.
- **`classes` computed once, globally**, then passed to `predict_model` per fold — that's
  what makes the zero-padding above work.
- **A fresh model per fold**; its validation predictions land in `oof[va]`. After the loop,
  every training row has exactly one prediction made by a model that never saw it — the
  **out-of-fold (OOF) predictions**. They are the honest currency of the whole library:
  fold metrics, blend weights and baseline comparisons are all computed on them.
- **`labels` kwarg**: only consumed by the logloss branch of `compute_metric`; harmless
  everywhere else, needed when a fold's *validation* split misses a class.
- **The `final` model is refit on 100% of the data.** The fold models estimate
  generalization; the final model is the one you deploy (more data → at least as good).

`TrainResult` carries all of it: `oof`, `fold_metrics`, the fold `models`, `final_model`,
`feature_names` (column order matters at inference!), and `model_type`.

In [12]:
from tabular_agent.models import train_cv

y = df["churn"]
oof_te, final_enc = oof_target_encode(df[["city"]], y, ["city"], folds)   # same folds!
X = pd.concat([df[["age", "income"]], oof_te], axis=1)
print("features:", list(X.columns))

res_lgbm = train_cv(X, y, folds, "binary", "logloss", model_type="lgbm")
res_xgb = train_cv(X, y, folds, "binary", "logloss", model_type="xgb",
                   params={"max_depth": 3, "learning_rate": 0.05})   # params passthrough
res_base = train_cv(X, y, folds, "binary", "logloss", model_type="baseline")

for name, r in [("lgbm", res_lgbm), ("xgb", res_xgb), ("baseline", res_base)]:
    print(f"{name:9s} fold logloss: {np.round(r.fold_metrics, 4)}  "
          f"mean={np.mean(r.fold_metrics):.4f}")

features: ['age', 'income', 'city_te']


lgbm      fold logloss: [0.6173 0.6608 0.604  0.6124 0.6025]  mean=0.6194
xgb       fold logloss: [0.5464 0.5627 0.4811 0.5301 0.5012]  mean=0.5243
baseline  fold logloss: [0.693  0.693  0.6929 0.6929 0.6929]  mean=0.6929


## 9. `blend` — ensemble weights fitted on OOF predictions

Blending averages several models' predictions with learned weights. The design questions
are *what data to fit the weights on* and *how to keep them valid*.

**Why OOF predictions?** Fitting weights on in-sample training predictions would reward the
most overfit model. The OOF predictions from section 8 are out-of-sample for *every* row,
so weights fitted on them are honest.

```python
def _normalize(w: np.ndarray) -> np.ndarray:
    w = np.clip(w, 0.0, None)
    total = w.sum()
    return w / total if total > 0 else np.ones(len(w)) / len(w)

def optimize_blend(oofs, y, metric, task_type) -> np.ndarray:
    sign = -1.0 if greater_is_better(metric) else 1.0

    def objective(raw):
        w = _normalize(raw)
        pred = sum(wi * o for wi, o in zip(w, oofs))
        return sign * compute_metric(metric, y, pred)

    result = minimize(objective, np.ones(m) / m, method="Nelder-Mead", ...)
    return _normalize(result.x)
```

- **`_normalize`**: Nelder-Mead searches unconstrained ℝᵐ, but every candidate is projected
  onto valid blend weights — clipped to non-negative, renormalized to sum to 1 (all-zero
  falls back to uniform). Constrained problem, unconstrained optimizer, no constraint
  handling needed.
- **`sign`**: `minimize` always minimizes; for greater-is-better metrics (AUC) the
  objective is negated. `greater_is_better` from section 7 supplies the orientation.
- **Nelder-Mead** is derivative-free — important because metrics like AUC are not smooth in
  the weights — and perfectly adequate in this tiny dimension (one weight per model).
- Convex combinations of probabilities are still probabilities, so the blended prediction
  respects the same format contract.

`apply_blend` replays the weighted sum at inference and validates that the number of
weights matches the number of prediction arrays.

In [13]:
from tabular_agent.blend import optimize_blend, apply_blend

oofs = [res_lgbm.oof, res_xgb.oof, res_base.oof]
weights = optimize_blend(oofs, y, "logloss", "binary")

for name, w, r in zip(["lgbm", "xgb", "baseline"], weights, [res_lgbm, res_xgb, res_base]):
    print(f"{name:9s} weight={w:.3f}   OOF logloss={compute_metric('logloss', y, r.oof):.4f}")

blend_oof = apply_blend(oofs, weights)
blend_score = compute_metric("logloss", y, blend_oof)
print(f"\nblended OOF logloss = {blend_score:.4f}")

lgbm      weight=0.183   OOF logloss=0.6194
xgb       weight=0.731   OOF logloss=0.5243
baseline  weight=0.086   OOF logloss=0.6929

blended OOF logloss = 0.5175


The optimizer puts most weight on the strongest model, keeps some of the second one for
diversity, and gives the uninformative baseline almost nothing — and the blend beats the
best single model on the OOF data it was fitted on.

## 10. `report` — honest reporting

`build_report` renders a markdown summary of a run: task, metric, CV score with **fold
standard deviation** (a stability indicator — a great mean with huge fold variance is not
trustworthy), baseline score, models, features, per-fold metrics.

Its most important feature is the **baseline-honesty warning**:

```python
def _beats_baseline(metric, selection_score, baseline_score) -> bool:
    if greater_is_better(metric):
        return selection_score >= baseline_score
    return selection_score <= baseline_score
```

If the selected model does not beat the trivial baseline, the report says so, loudly —
because that outcome almost always means a bug, leakage, or an unlearnable target, and an
agent (or human) skimming the report must not miss it. Note the orientation-awareness:
"beats" means *higher* for AUC but *lower* for logloss/RMSE.

`write_report` saves the text as `run_report.md` inside the run directory — one more
artifact in the run-dir-is-the-state model.

In [14]:
from tabular_agent.report import build_report, write_report

baseline_score = compute_metric("logloss", y, res_base.oof)

report = build_report(
    task_type="binary",
    metric="logloss",
    selection_score=blend_score,
    fold_metrics=res_xgb.fold_metrics,     # fold metrics of the strongest single model
    baseline_score=baseline_score,
    model_types=["lgbm", "xgb", "baseline"],
    feature_names=list(X.columns),
)
print(report)

# Run Report

- **Task type:** binary
- **Selection metric:** logloss
- **CV score:** 0.518 (fold std 0.030)
- **Baseline score:** 0.693
- **Models:** lgbm, xgb, baseline
- **Features (3):** age, income, city_te

## Per-fold metrics
- fold 0: 0.5464
- fold 1: 0.5627
- fold 2: 0.4811
- fold 3: 0.5301
- fold 4: 0.5012


In [15]:
# Same call, but pretend our model is WORSE than the baseline (higher logloss):
bad = build_report(
    task_type="binary",
    metric="logloss",
    selection_score=baseline_score + 0.05,
    fold_metrics=res_xgb.fold_metrics,
    baseline_score=baseline_score,
    model_types=["lgbm"],
    feature_names=list(X.columns),
)
print("\n".join(line for line in bad.splitlines() if "WARNING" in line))

report_path = write_report(run_ctx, report)
print("\nreport written to:", report_path)

> **WARNING:** the model does not beat the trivial baseline. Suspect a bug, leakage, or an unlearnable target.

report written to: /var/folders/xr/7r1214b13cx19cv3dt5j48mc0000gn/T/tmp0wtbl7wb/run-001/run_report.md


## 11. `pipeline` — a predictor you can ship

The last mile: package everything needed for inference so that `predict` works on raw data
**without the target column and without re-running any training code**.

### `Pipeline` — feature steps as data, not objects

```python
def add_encoder(self, encoder) -> None:
    self.steps.append({"type": "encode", "encoder": encoder.to_dict()})

def transform(self, df):
    for step in self.steps:
        enc = ENCODER_CLASSES[step["encoder"]["kind"]].from_dict(step["encoder"])
        out = enc.transform(out)
```

Steps are stored as **plain dicts** (via `to_dict` from section 6), not Python objects.
That makes the whole pipeline JSON-serializable; `transform` rehydrates the right encoder
class through the `ENCODER_CLASSES` registry and replays its frozen parameters. No pickle
needed for the feature logic — it's inspectable text.

### `save_bundle` / `load_bundle` / `predict_bundle`

A bundle directory contains:

- `model_{i}.joblib` — the fitted final models (joblib because sklearn-style estimators
  aren't JSON-serializable),
- `pipeline.json` — the manifest: pipeline dict, model paths, blend weights,
  `feature_names`, `task_type`.

```python
def predict_bundle(bundle, raw_df):
    features = bundle["pipeline"].transform(raw_df)     # replay frozen encoders
    X = features[bundle["feature_names"]]               # exact training columns, exact order
    preds = [predict_model(m, X, task_type) for m in bundle["models"]]
    return apply_blend(preds, bundle["weights"])
```

Two details matter:

- **`feature_names` selection**: the raw frame may contain extra columns (IDs, etc.) and
  `transform` appends columns; selecting by the stored list guarantees each model sees
  exactly the columns it was trained on, in the same order.
- **The OOF/final asymmetry from section 6 resolves here**: models were *trained* on
  OOF-encoded features (honest evaluation), but at *inference* the pipeline applies the
  final encoder fit on all training rows (best available estimates). This is the standard,
  correct pattern for target encoding.

In [16]:
from tabular_agent.pipeline import Pipeline, save_bundle, load_bundle, predict_bundle

pipe = Pipeline("binary")
pipe.add_encoder(final_enc)                     # the full-data encoder from section 8

models_final = [res_lgbm.final_model, res_xgb.final_model, res_base.final_model]
bundle_dir = save_bundle(run_ctx, pipe, models_final, weights, list(X.columns), "binary")
print("bundle contents:", sorted(os.listdir(bundle_dir)))

bundle contents: ['model_0.joblib', 'model_1.joblib', 'model_2.joblib', 'pipeline.json']


In [17]:
bundle = load_bundle(bundle_dir)

new_rows = df.drop(columns="churn").head(5)     # raw data, no target column
preds = predict_bundle(bundle, new_rows)
print("predicted churn probabilities:", np.round(preds, 3))

# sanity check: identical to predicting with the in-memory objects
feats = pipe.transform(new_rows)
manual = apply_blend(
    [predict_model(m, feats[list(X.columns)], "binary") for m in models_final], weights
)
print("round-trip identical:", np.allclose(preds, manual))

predicted churn probabilities: [0.762 0.576 0.625 0.378 0.09 ]
round-trip identical: True


## 12. End to end — the whole workflow in one cell

Everything above, chained. This is exactly the sequence the Phase-2 MCP server will
orchestrate tool-by-tool, with each intermediate artifact living in the run directory.

In [18]:
# 1. load + inspect
data = load_table(path)                                        # the parquet from section 1
spec = resolve_taskspec(TaskSpec(target="churn"), data["churn"])

# 2. folds (shared by encoding and training)
y_all = data[spec.target]
fs = make_folds(y_all, spec.task_type, spec.n_splits, spec.seed)

# 3. leakage-safe features
te_oof, te_final = oof_target_encode(data[["city"]], y_all, ["city"], fs)
X_all = pd.concat([data[["age", "income"]], te_oof], axis=1)

# 4. train baseline + two models
results = {mt: train_cv(X_all, y_all, fs, spec.task_type, spec.metric, mt, params=p)
           for mt, p in [("baseline", None), ("lgbm", None),
                         ("xgb", {"max_depth": 3, "learning_rate": 0.05})]}

# 5. blend on OOF predictions
w = optimize_blend([results["lgbm"].oof, results["xgb"].oof], y_all,
                   spec.metric, spec.task_type)
blended = apply_blend([results["lgbm"].oof, results["xgb"].oof], w)

# 6. report
ctx = RunContext(os.path.join(tmp, "run-e2e"))
txt = build_report(
    task_type=spec.task_type, metric=spec.metric,
    selection_score=compute_metric(spec.metric, y_all, blended),
    fold_metrics=results["lgbm"].fold_metrics,
    baseline_score=compute_metric(spec.metric, y_all, results["baseline"].oof),
    model_types=["lgbm", "xgb"], feature_names=list(X_all.columns),
)
write_report(ctx, txt)

# 7. ship + predict
pl = Pipeline(spec.task_type)
pl.add_encoder(te_final)
bd = save_bundle(ctx, pl, [results["lgbm"].final_model, results["xgb"].final_model],
                 w, list(X_all.columns), spec.task_type)
p_new = predict_bundle(load_bundle(bd), data.drop(columns="churn").tail(3))

print(txt.split("## Per-fold")[0])
print("blend weights:", np.round(w, 3))
print("predictions on new rows:", np.round(p_new, 3))

# Run Report

- **Task type:** binary
- **Selection metric:** logloss
- **CV score:** 0.522 (fold std 0.021)
- **Baseline score:** 0.693
- **Models:** lgbm, xgb
- **Features (3):** age, income, city_te


blend weights: [0.144 0.856]
predictions on new rows: [0.25  0.693 0.588]


## What Phase 2 adds

Phase 1 is a library: honest primitives with explicit data flow. Phase 2 (`mcp-surface`)
wraps these functions as MCP tools — roughly `inspect_data`, `train`, `blend`, `report`,
`predict` — so an agent can drive the workflow conversationally. The design decisions you
saw here are what make that safe:

- **`RunContext`**: every tool call is stateless; the run directory carries all state.
- **Shared folds**: the agent cannot accidentally create leakage by calling tools in a
  strange order — folds are created once and reused.
- **Serializable everything** (`TaskSpec`, encoder dicts, `pipeline.json`): tool inputs and
  outputs are plain data.
- **The baseline warning**: the agent gets an explicit signal when results are too bad to
  ship — honesty is built into the report, not left to the agent's judgment.